# CelerisAi Web-Config Tutorial

This notebook reproduces the `setrun_web.py` workflow using CelerisWEBGPU-compatible case folders.

What you will do:
- Read topography/config from a case folder
- Build domain and boundary conditions in `celeris=True` mode
- Run a 2D simulation with live display

## Prerequisites
If you have not installed CelerisAi yet, you can do it just with:

From repository root:
```bash
pip install -e .
```

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

## 1) Imports and backend

In [ ]:
import taichi as ti

from celeris.domain import Topodata, BoundaryConditions, Domain
from celeris.solver import Solver
from celeris.runner import Evolve

ti.init(arch=ti.gpu)

## 2) Select a case folder

Expected case files include `config.json`, `bathy.txt`, and usually `waves.txt`.

In [ ]:
EXAMPLES_DIR = repo_root / 'examples'
CASE = 'Balboa'
CASE_PATH = EXAMPLES_DIR / CASE

CASE_PATH

## 3) Build model inputs

- `Topodata(datatype='celeris')` reads CelerisWebGPU-style bathymetry
- `BoundaryConditions(celeris=True)` reads `config.json` in the case folder

In [ ]:
precision = ti.f32

baty = Topodata(datatype='celeris', path=str(CASE_PATH))
bc = BoundaryConditions(celeris=True, path=str(CASE_PATH), precision=precision)
d = Domain(topodata=baty, precision=precision)

d.Nx, d.Ny

## 4) Configure and run solver

In [ ]:
solver = Solver(domain=d, boundary_conditions=bc, model='Bouss')

run = Evolve(
    solver=solver,
    maxsteps=3000,
    saveimg=True,
    plot_interval=100
)
# Interactive 2D display
#run.Evolve_Display(variable='h', cmapWater='Blues')

# Headless alternative:
run.Evolve_Headless()

## Notes

- Change `CASE` to run another folder under `examples/` (for example `Ventura` or `CrescentCity`).
- Use `saveimg=False` for lighter runs.
- For CPU execution, switch to `ti.init(arch=ti.cpu)`.

## 5 Plot the results (last step)
This process can be done in any step of the numerical simulation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# extract the coordinates
x_coord,y_coord,zz =solver.domain.grid()
# Hide eta where zz <= 0 (dry land)
mask = zz <= 0 

#Send data to numpy
q = run.solver.State.to_numpy()
eta_ = q[:,:,0]
hu = q[:,:,1]
# Apply the mask to water data
eta_masked = np.ma.masked_where(mask, eta)
hu_masked = np.ma.masked_where(mask, hu)
# Plot
plt.pcolor(x_coord.T,y_coord.T,eta_masked,cmap='Blues',zorder=1)
#plt.pcolor(x_coord.T,y_coord.T,hu_masked,cmap='jet',zorder=1)
plt.pcolor(x_coord.T,y_coord.T,-zz,cmap='gist_earth',zorder=0)# make bathymetry negative
